# Graph representation learning
Dado um grafo de entrada, extrair features de nó, link e nível de grafo, e então aprender um modelo (SVM, rede neural, etc.) que mapeia características para labels.

Graph representation learning alivia a necessidade de fazer feature engineering toda vez. Graph representation learning automaticamente aprende as features

# Node embedding
Objetivo do embedding é mapear os nodes em um embedding space
- Similaridade de embeddings entre nodes indica a similaridade deles na network, por exemplo:
    - Ambos os nodes estão pertos um do outro (conectados por um edge)
    - Transformar informações de um grafo em uma representação numérica ou vetorial
    - A representação aprendida pode ser reutilizada para vários tipos de tarefas posteriores de previsão.

![Node embedding usage](images/node_embedding.png)

## Exemplo de node embedding:
![Node embedding example](images/node_embedding_example.png)

## Node embedding: Encoder e Decoder
### Setup
Assumimos que temos um grafo não direcionado $G$:
- $V$ é o vertex set
- $A$ é a matriz de abjacência (assume-se binário)
- Para simplicidade: Sem node features ou informações extras estão sendo usadas
![G Graph](images/graph_G.png)

O objetivo para encode nodes para que a similaridade no embeddding space aproxime a similaridade no grafo
![Embedding Space](images/emb_space.png)

1. O encoder mapeia dos nodes para o embedding
2. Define a função de similaridade do node (ex: medida da similaridade na network original)
3. Decoder $DEC$ mapeia dos embeddings para o score da similaridade
4. Otimiza parâmetros do encoder para que a similaridade na network original aproxime a similaridade do embedding

### Dois key components
- Encoder mapeia cada node para um vetor de baixa dimensão
- Função de similaridade: Especifica como as relações no vector space mapeiam para as relações na network original

### "Shallow" encoding
Approach mais simples: Encoder é apenas um embedding lookup
![Shallow encoding](images/shallow_encoding.png)
![Shallow encoding](images/shallow_encoding_1.png)

Cada node é assimilado à um único embedding vector (ex: nós diretamente otimizamos o embedding de cada node). Isso tem vários métodos: DeepWalk, node2vec

### Sumário do Framework
- Encoder + decoder framework
    - Shallow encoder: Lookup do embedding
    - Parâmetros para otimizar: $Z$ que contém node embeddings $z_u$ para todos os nodes
    - Decoder: baseado na similaridade de node

### Como definir Node similarity?
Primeiro de tudo, Node embedding é uma maneira unsuperviserd/self-supervised de aprendizado. Não estamos utilizando node labels, nem node features. O objetivo é diretamente estimar um set de coordenadas de um node para que alguns aspectos da estrutura da network (capturado por DEC) é preservado

Esses embeddings são task independent:
 - Eles não são treinados para uma tarefa específica mas consegue ser usado para qualquer tarefa.

## Random Walk para node embeddings
### Notação
- Vetor $z_u$
    - O embedding do node $u$ (o que desejamos encontrar)
- Probailidade  $P(v|z_u)$:
    - A probabilidade (predita) do node visitante $v$ em random walks começando do node $u$.
---
Funções não lineares usadas para produzir probabilidades preditas
- Softmax: Transforma vetor de $K$ valores reais (predições do modelo) em $K$ probabilidades que somam 1
- Sigmoid: Função com formato em S que transforma valores reais em um range de (0,1)

## Random walk
Dado um grafo e um ponto inicial, selecionamos um vizinho dele aleatoriamente e nos movemos para esse vizinho; então selecionamos um vizinho desse novo ponto aleatoriamente e nos movemos para ele, e assim por diante.

![Random walk](images/random_walk.png)

1. Expressividade: definição estocástica flexível de similaridade entre nós, que incorpora tanto informação da vizinhança local quanto informação de vizinhança de ordem superior.

2. Ideia: se uma random walk começando no nó $u$ visita $v$ com alta probabilidade, então $u$ e $v$ são considerados similares (captura informação multi-hop de ordem superior).

3. Eficiência: não é necessário considerar todos os pares de nós durante o treinamento; basta considerar pares que aparecem juntos nas random walks.

## Unsupervised features learning
- Intuição: Encontrar embedding de nodes no espaço $d$-dimensional que preserva a similaridade
- Ideia: Aprender node embedding tanto que nodes pertos estão juntos na network
- Dado o node $u$, como definimos nodes próximos?
    - $N_R(u)$ ... vizinhança de $u$ obtida por algumas random walks strategy $R$

## Sumário de random walks
Execute random walks curtas de comprimento fixo, começando de cada nó do grafo.

Para cada nó $u$, colete $N_R(u)$, o multiconjunto de nós visitados nas random walks que começam em $u$.

Otimize os embeddings $Z$ utilizando Stochastic Gradient Descent (SGD).

# node2vec
- Objetivo: Embed nodes com vizinhos de network similares pertos no feature space.

- Nós enquadramos o objetivo como um problema de otimização de probabilidade máxima, independete da tarefa de predição

- Observação chave: Uma noção flexível de vizinhança de rede $N_R(u)$ do node $u$ leva à geração de embeddings de nodes mais ricos.

## Walks enviesadas
- Ideia: Usar random walks flexíveis, enviesadas que consegue trocar entre local e global views da network

BFS (Breadth-First Search) vê a estrutura local
DFS (Depth-Fisrt Search) vê a estrutura global

![DFS BFS](images/bfs_dfs.png)

- Random walk têm dois parâmetros
    - Retorno parâmetro $p$:
        - Retorna para o node anterior
    - Entrada-saída parâmetro $q$
        - Movendo para fora (DFS) vs. para dentro (BFS) do node anterior
        - Intuitivamente, $q$ é o "ratio"de BFS vs. DFS

### Algoritmo node2vec
1. Computar a prob. de transição entre edges
2. Simula $r$ random walks de tamanho $l$ começando de cada node $u$
3. Otimiza o node2vec usando SGD